# Training the CNN model

In [ ]:
# Raíz del repo e imports
import sys
from pathlib import Path
ROOT_ = Path.cwd().resolve()
while not (ROOT_ / "requirements.txt").exists():
    ROOT_ = ROOT_.parent
sys.path.insert(0, str(ROOT_))

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from src.utils.config import (SEED, CLASSES, CLASS_FOLDERS, PROCESSED_DIR,
                              MODELS_DIR, RESULTS_DIR, CNN_IMG_SIZE)
from src.utils.evaluation import save_predictions
from src.models.cnn import build_cnn_model

tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = CNN_IMG_SIZE   # ahora viene de config: (224, 224)
BATCH = 32

# > **Pendiente: agregar explicación del pipeline de datos, data augmentation y decisiones de entrenamiento.**

In [ ]:
# Tabla de rutas y etiquetas (PROVISIONAL: solo las clases que existan)
rows = []
for idx, cls in enumerate(CLASSES):
    folder = PROCESSED_DIR / CLASS_FOLDERS[cls]
    if folder.exists():
        for p in folder.rglob("*"):
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                rows.append({"path": str(p), "label": idx})
df = pd.DataFrame(rows)
print(df.label.map(dict(enumerate(CLASSES))).value_counts())

# Reemplazar por el compartido
df_train, df_tmp = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
df_val, df_test = train_test_split(df_tmp, test_size=0.50, stratify=df_tmp.label, random_state=SEED)
# df_test NO se toca hasta el final.

In [ ]:
# Pipeline tf.data
def make_ds(frame, shuffle):
    def load(path, y):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        return tf.image.resize(img, IMG_SIZE), y
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.label.values))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

ds_train = make_ds(df_train, shuffle=True)
ds_val   = make_ds(df_val, shuffle=False)

In [ ]:
# Modelo, pesos de clase y entrenamiento
model = build_cnn_model(input_shape=(*IMG_SIZE, 3), num_classes=len(CLASSES))
model.summary()

classes_present = np.unique(df_train.label)
weights = compute_class_weight("balanced", classes=classes_present, y=df_train.label)
class_weight = dict(zip(classes_present, weights))

MODELS_DIR.mkdir(parents=True, exist_ok=True)
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(MODELS_DIR / "cnn_best.keras", monitor="val_loss", save_best_only=True),
]
history = model.fit(ds_train, validation_data=ds_val, epochs=25,
                    class_weight=class_weight, callbacks=callbacks)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(history.history).to_csv(RESULTS_DIR / "cnn_history.csv", index=False)

In [ ]:
# Predicciones de validación en el formato común
proba = model.predict(ds_val)               
y_pred = [CLASSES[i] for i in proba.argmax(axis=1)]
y_true = [CLASSES[i] for i in df_val.label]
save_predictions(df_val.path.values, y_true, y_pred, proba, model_name="cnn", split="val")